# News Discovery System — Production Colab Run + Validation Harness

This notebook launches the **real in-repo Gradio analyst UI** and provides notebook-level diagnostics for validation when UI runs fail.


## 1) Environment setup
- Detect Colab runtime and Python version.
- Define a stable project location used across notebook cells.


In [1]:
from pathlib import Path
import os
import platform

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
PROJECT_DIR = Path("/content/news-discovery-system") if IN_COLAB else Path.cwd()
os.environ.setdefault("MPLBACKEND", "Agg")

print(f"Python: {platform.python_version()}")
print(f"In Colab: {IN_COLAB}")
print(f"Project directory target: {PROJECT_DIR}")


Python: 3.12.13
In Colab: True
Project directory target: /content/news-discovery-system


## 2) Dependency installation
Installs runtime dependencies that are actually used by this repository (`gradio`, `matplotlib`, `plotly`, `requests`) plus `pytest` for smoke checks.


In [2]:
import subprocess
import sys

RUNTIME_PACKAGES = ["gradio", "matplotlib", "plotly", "requests", "pytest==9.0.2"]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PACKAGES])

print("Installed packages:")
for package in RUNTIME_PACKAGES:
    print(f" - {package}")


Installed packages:
 - gradio
 - matplotlib
 - plotly
 - requests
 - pytest==9.0.2


## 3) Repository load / path setup
- Uses current directory if the notebook is already inside the repository.
- Otherwise clones the repository from `REPO_URL`.


In [3]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.getenv("NEWS_DISCOVERY_REPO_URL", "https://github.com/dshipley71/news-discovery-system.git")

if (Path.cwd() / "gr_app.py").exists():
    PROJECT_DIR = Path.cwd()
    print(f"Using existing repository at: {PROJECT_DIR}")
else:
    if "<your-org>" in REPO_URL:
        raise ValueError(
            "Set NEWS_DISCOVERY_REPO_URL to your repository URL before running this cell. "
            "Example: os.environ['NEWS_DISCOVERY_REPO_URL'] = 'https://github.com/my-org/news-discovery-system.git'"
        )

    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)

    subprocess.check_call(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    os.chdir(PROJECT_DIR)
    print(f"Cloned repository to: {PROJECT_DIR}")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

required_paths = [PROJECT_DIR / "gr_app.py", PROJECT_DIR / "src" / "news_app" / "workflow.py", PROJECT_DIR / "config" / "sources.json"]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("Repository path setup complete.")


Cloned repository to: /content/news-discovery-system
Repository path setup complete.


## 4) Optional configuration / secrets guidance
The X/Twitter source is optional. If `TWITTER_BEARER_TOKEN` is missing, workflow execution continues with a source warning.


In [4]:
import os

# Optional: uncomment and set manually if needed.
# os.environ["TWITTER_BEARER_TOKEN"] = "YOUR_TOKEN"

twitter_token = os.getenv("TWITTER_BEARER_TOKEN", "").strip()
if twitter_token:
    print("TWITTER_BEARER_TOKEN is set. X/Twitter adapter will be attempted.")
else:
    print("TWITTER_BEARER_TOKEN is not set. X/Twitter will be skipped with explicit warnings.")


TWITTER_BEARER_TOKEN is not set. X/Twitter will be skipped with explicit warnings.


## 5) Launch Gradio UI (`share=True`)
This cell runs `gr_app.py` from the repository and keeps it running in the background so other validation cells remain usable.


In [5]:
import sys
import importlib
from pathlib import Path

repo_path = str(PROJECT_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if "gr_app" in sys.modules:
    del sys.modules["gr_app"]

import gr_app
importlib.reload(gr_app)

demo = gr_app.build_app()
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    debug=True,
    inline=False,
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bda4d8b0ef8d559c92.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://bda4d8b0ef8d559c92.gradio.live


## 6) Optional debug / smoke-test cells
Use these cells to validate backend execution and inspect intermediate artifacts without leaving Colab.


In [ ]:
from datetime import datetime, timedelta, timezone
import json

from src.news_app.workflow import RunInput, run_workflow

# Keep range short for faster diagnostics
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=2)

smoke_input = RunInput(topic="global supply chain", start_date=start_date, end_date=end_date)
smoke_result = run_workflow(smoke_input)

print("Run ID:", smoke_result["run_id"])
print("Input:", smoke_result["input"])
print("Sources attempted:", smoke_result["stages"]["ingestion"]["sources_attempted"])
print("Sources succeeded:", smoke_result["stages"]["ingestion"]["sources_succeeded"])
print("Sources failed:", smoke_result["stages"]["ingestion"]["sources_failed"])
print("Canonical article count:", smoke_result["stages"]["normalization"]["valid_count"])
print("Duplicate ratio:", smoke_result["stages"]["ingestion"]["telemetry"]["ingestion_duplicate_ratio"])
print("Cluster count:", smoke_result["stages"]["clustering"]["cluster_count"])
print("Geospatial marker count:", len(smoke_result["stages"]["geospatial"]["map_markers"]))
print("Timeline day count:", smoke_result["stages"]["aggregation"]["total_days"])
print("Citation count:", smoke_result["stages"]["citation_traceability"]["citation_count"])
print("Warning count:", len(smoke_result["stages"]["warnings"]))


In [ ]:
from pprint import pprint

def diagnostics(result: dict) -> dict:
    ingestion = result["stages"]["ingestion"]
    normalization = result["stages"]["normalization"]
    aggregation = result["stages"]["aggregation"]
    clustering = result["stages"]["clustering"]
    geospatial = result["stages"]["geospatial"]
    citations = result["stages"]["citation_traceability"]

    timeline = aggregation["daily_counts"]
    timeline_is_sorted = timeline == sorted(timeline, key=lambda row: row["day"])

    summary = {
        "per_source_ingestion": ingestion["source_runs"],
        "partial_failures": [run for run in ingestion["source_runs"] if run["status"] != "success"],
        "normalization_output": {
            "valid_count": normalization["valid_count"],
            "invalid_count": normalization["invalid_count"],
        },
        "duplicate_handling": {
            "duplicate_ratio": ingestion["telemetry"].get("ingestion_duplicate_ratio"),
            "duplicate_groups": len(ingestion["telemetry"].get("duplicate_map", [])),
        },
        "clustering_output": {
            "cluster_count": clustering["cluster_count"],
            "sample_clusters": clustering["clusters"][:2],
        },
        "geospatial_output": {
            "entity_count": geospatial["entity_count"],
            "marker_count": len(geospatial["map_markers"]),
        },
        "timeline_correctness": {
            "day_count": aggregation["total_days"],
            "is_sorted": timeline_is_sorted,
            "sample_days": timeline[:5],
        },
        "citation_evidence": {
            "citation_count": citations["citation_count"],
            "bundle_count": len(result["stages"]["evidence"].get("cluster_to_articles", [])),
        },
        "warning_behavior": result["stages"]["warnings"],
    }
    return summary

summary = diagnostics(smoke_result)
pprint(summary)


## 7) Analyst validation instructions (UI-first)
1. Open the public Gradio URL from Section 5.
2. Run a workflow using a topic and a date window (max 30 days).
3. Confirm all major outputs are inspectable in the UI:
   - per-source ingestion status and counts
   - partial source failures and warning visibility
   - normalization records and validation issues
   - duplicate handling signals
   - clustering summary and detail
   - geospatial markers/entity confidence
   - timeline chart and day-level counts
   - citation index and evidence bundles
4. If UI output looks wrong or empty, return to Section 6 smoke tests and compare stage payloads.
5. Keep the notebook open during review so the shared Gradio URL remains active.
